# Решения из гауссова импульса

При каких амплитудах и ширинах начальной гауссианы к $T=100$ наблюдаются однородное решение, одиночный солитон или прочее состояние?
Начальный импульс:
$$\psi(\theta,0)=A\exp\left[-\frac{\theta^2}{2\sigma^2}\right].$$
Параметры уравнения: $F=2$, $\Delta=3$, $\beta_2=-0.1$, $L=2\pi$.
Сетка начальных условий: $8\times8$, $A\in[0.4,3.2]$ равномерно,
$\sigma\in[0.05,0.7]$ логарифмически. Расчёты: $N=512$,
$\Delta t=0.0005$, интервал записи $0.5$.


In [ ]:
from enum import Enum, auto
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import LogNorm
from lle_runs.project import get_experiment_dir
from lle_analysis.run_selection import load_runs_from_csv
from lle_solver.simulations.gaussian import GaussianConfig
from lle_solver.simulations.storage import load_evolution
from lle_analysis.criteria import (
    DEFAULT_STATIONARY_TOLERANCE, DEFAULT_HOMOGENEOUS_CONTRAST, is_stationary, is_homogeneous,
)
from lle_analysis.fields import time_mask, relative_field_l2_error, spatial_contrast
from lle_analysis.peaks import peak_indices
from lle_viz.style import configure_plots, save_figure, format_log_x
from lle_viz.plots import cell_edges, plot_outcome_map
from lle_viz.grids import rectangular_grid

In [ ]:
EXPERIMENT_DIR = get_experiment_dir(1)
RESULTS_DIR = EXPERIMENT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
configure_plots()

## Исходная сетка

[Запуски исходной сетки](coarse_runs.csv): 64 расчёта.


In [ ]:
coarse_runs = load_runs_from_csv(EXPERIMENT_DIR / "coarse_runs.csv")

In [ ]:
coarse_configs = {run.id: run.load_config(GaussianConfig) for run in coarse_runs}
coarse_evolutions = {run.id: load_evolution(run.path) for run in coarse_runs}

### Изменение поля и контраст на сетке начальных условий

Для каждого узла берём фиксированный интервал $[80,100]$. Левая карта показывает максимальное относительное изменение комплексного поля, правая — максимальный пространственный контраст интенсивности. Величины разделены на соответствующие пороги: $10^{-3}$ и $0.01$. Синий цвет означает значение ниже порога, красный — выше; значение 1 соответствует порогу.


In [ ]:
observation_time = 100.
observation_duration = 20.
tails = {}
measurements = []
for run in coarse_runs:
    initial = coarse_configs[run.id].initial
    evolution = coarse_evolutions[run.id]
    tail = evolution[time_mask(
        evolution.times, observation_time - observation_duration, observation_time,
    )]
    if (tail.times.size < 2
            or not np.isclose(tail.times[0], observation_time - observation_duration, rtol=0, atol=1e-10)
            or not np.isclose(tail.times[-1], observation_time, rtol=0, atol=1e-10)):
        raise ValueError(f"Запуск {run.id} не покрывает окно наблюдения")
    tails[run.id] = tail
    measurements.append(dict(A=initial.amplitude, sigma=initial.width,
                             relative_field_l2_error=float(relative_field_l2_error(tail, tail[-1]).max()),
                             contrast=float(spatial_contrast(tail.intensity).max()), run_id=run.id))

measurements.sort(key=lambda row: (row["A"], row["sigma"]))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout="constrained")
metrics = (
    ("relative_field_l2_error", DEFAULT_STATIONARY_TOLERANCE, "Изменение комплексного поля"),
    ("contrast", DEFAULT_HOMOGENEOUS_CONTRAST, "Контраст интенсивности"),
)
for ax, (metric, threshold, title) in zip(axes, metrics):
    widths, amplitudes, values = rectangular_grid(
        [row["sigma"] for row in measurements],
        [row["A"] for row in measurements],
        [row[metric] for row in measurements],
    )
    ratio = values / threshold
    image = ax.pcolormesh(
        cell_edges(widths, logarithmic=True), cell_edges(amplitudes),
        np.maximum(ratio, np.finfo(float).tiny),
        norm=LogNorm(vmin=1e-3, vmax=1e3), cmap="RdBu_r", rasterized=True,
    )
    x, y = np.meshgrid(widths, amplitudes)
    ax.scatter(x.ravel(), y.ravel(), s=9, color="black", alpha=.5)
    ax.set(xscale="log", xlabel=r"Ширина $\sigma$", title=title)
    format_log_x(ax)
axes[0].set_ylabel(r"Амплитуда $A$")
colorbar = figure.colorbar(image, ax=axes, extend="both", ticks=[1e-3, 1e-2, 1e-1, 1, 10, 100, 1000])
colorbar.set_label("Значение / порог")
colorbar.ax.axhline(1, color="black", linewidth=1)
figure.suptitle(r"Измерения на интервале $t \in [80, 100]$")
save_figure(RESULTS_DIR, "measurement_maps", figure)
plt.show()

### Три категории на интервале $[80,100]$

- **Однородное решение:** комплексное поле стационарно, интенсивность однородна на каждом снимке.
- **Одиночный солитон:** комплексное поле стационарно, интенсивность неоднородна и имеет ровно один выраженный пик на каждом снимке.
- **Прочее:** остальные состояния; категория не определяет характер динамики.


In [ ]:
class SolutionKind(Enum):
    HOMOGENEOUS = auto()
    SINGLE_SOLITON = auto()
    OTHER = auto()


coarse = []
for row in measurements:
    tail = tails[row["run_id"]]
    stationary = is_stationary(tail)
    intensity = tail.intensity
    solution = SolutionKind.OTHER
    if stationary:
        if all(is_homogeneous(profile) for profile in intensity):
            solution = SolutionKind.HOMOGENEOUS
        elif all(not is_homogeneous(profile) and len(peak_indices(profile)) == 1
                 for profile in intensity):
            solution = SolutionKind.SINGLE_SOLITON
    coarse.append(dict(A=row["A"], sigma=row["sigma"], solution=solution, run_id=row["run_id"]))

### Карта при $T=100$

Точки отмечают рассчитанные узлы. Цвет ячейки не устанавливает исход во всей её площади.


In [ ]:
solution_labels = {
    SolutionKind.HOMOGENEOUS: "Однородное решение",
    SolutionKind.SINGLE_SOLITON: "Одиночный солитон",
    SolutionKind.OTHER: "Прочее",
}
solutions = list(solution_labels)
colors = ["#293142", "#4fc48b", "#b580df"]

In [ ]:
widths, amplitudes, matrix = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [solutions.index(row["solution"]) for row in coarse],
)

In [ ]:
figure, ax = plt.subplots(figsize=(6.4, 6.1))
image, nodes = plot_outcome_map(ax, widths, amplitudes, matrix, colors)
nodes.set(sizes=[10], color="black", alpha=.5)
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$", title="Решения при $T=100$")
format_log_x(ax)
figure.legend(handles=[Patch(color=color, label=solution_labels[solution])
                       for color, solution in zip(colors, solutions)],
              loc="lower center", frameon=False)
figure.tight_layout(rect=(0, .20, 1, 1))
save_figure(RESULTS_DIR, "outcome_map", figure)
plt.show()

### Начальная энергия и наблюдаемое решение

Для гауссова импульса используем аналитическую формулу $E_0=\sqrt{\pi}A^2\sigma$, без численного интегрирования. Поправкой конечного интервала пренебрегаем: при $L=2\pi$ и $\sigma\leq0.7$ она мала.


In [ ]:
energy_widths, energy_amplitudes, initial_energy = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [np.sqrt(np.pi) * row["A"]**2 * row["sigma"] for row in coarse],
)
energy_norm = LogNorm(vmin=initial_energy.min(), vmax=initial_energy.max())
energy_cmap = plt.get_cmap("viridis")
solution_markers = {
    SolutionKind.HOMOGENEOUS: "o",
    SolutionKind.SINGLE_SOLITON: "*",
    SolutionKind.OTHER: "^",
}
figure, ax = plt.subplots(figsize=(9, 8))
image = ax.pcolormesh(
    cell_edges(energy_widths, logarithmic=True), cell_edges(energy_amplitudes),
    initial_energy, cmap=energy_cmap, norm=energy_norm, rasterized=True,
)
for solution, marker in solution_markers.items():
    rows = [row for row in coarse if row["solution"] is solution]
    ax.scatter([row["sigma"] for row in rows], [row["A"] for row in rows],
               marker=marker, s=80 if marker == "*" else 35,
               facecolors="white", edgecolors="black", linewidths=.8,
               label=solution_labels[solution])
for row in coarse:
    energy = np.sqrt(np.pi) * row["A"]**2 * row["sigma"]
    red, green, blue, _ = energy_cmap(energy_norm(energy))
    luminance = .2126 * red + .7152 * green + .0722 * blue
    ax.annotate(f"{energy:.3g}", (row["sigma"], row["A"]),
                xytext=(0, -9), textcoords="offset points", ha="center", va="top",
                fontsize=9, color="white" if luminance < .5 else "black")
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$",
       title=r"Начальная энергия $E_0=\sqrt{\pi}A^2\sigma$")
format_log_x(ax)
colorbar = figure.colorbar(image, ax=ax, pad=.03)
colorbar.set_label(r"Начальная энергия $E_0$")
handles, labels = ax.get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", frameon=False, ncol=3, fontsize=10)
figure.tight_layout(rect=(0, .065, 1, 1))
save_figure(RESULTS_DIR, "initial_energy_map", figure)
plt.show()


### Профили интенсивности для всех начальных импульсов

В каждом узле показаны $|\psi(\theta,t)|^2$ при $t=0,20,50,100$. Столбцы соответствуют ширине $\sigma$, строки — амплитуде $A$ (большие амплитуды сверху, как на карте решений). Масштабы координаты и интенсивности одинаковы для всех 64 графиков. Последний профиль выделен более толстой линией.


In [ ]:
profile_times = (0., 20., 50., 100.)
profile_colors = ("#8b94a9", "#e69f00", "#0072b2", "#b2182b")
profile_styles = ("--", "-", "-", "-")
profile_amplitudes = sorted({config.initial.amplitude for config in coarse_configs.values()}, reverse=True)
profile_widths = sorted({config.initial.width for config in coarse_configs.values()})
profiles_by_node = {
    (coarse_configs[run.id].initial.amplitude, coarse_configs[run.id].initial.width):
        [coarse_evolutions[run.id].at_time(time).intensity for time in profile_times]
    for run in coarse_runs
}
assert len(profiles_by_node) == len(profile_amplitudes) * len(profile_widths)
maximum_intensity = max(profile.values.max() for profiles in profiles_by_node.values() for profile in profiles)

figure, axes = plt.subplots(
    len(profile_amplitudes), len(profile_widths), figsize=(20, 17),
    sharex=True, sharey=True, squeeze=False,
)
for row, amplitude in enumerate(profile_amplitudes):
    for column, width in enumerate(profile_widths):
        ax = axes[row, column]
        profiles = profiles_by_node[amplitude, width]
        for time, profile, color, style in zip(profile_times, profiles, profile_colors, profile_styles):
            ax.plot(profile.theta, profile.values, color=color, linestyle=style,
                    linewidth=1.5 if time == profile_times[-1] else .9, label=f"t={time:g}")
        ax.set_xlim(-np.pi, np.pi)
        ax.set_ylim(0, 1.05 * maximum_intensity)
        ax.set_xticks([-np.pi, 0, np.pi], labels=[r"$-\pi$", "0", r"$\pi$"])
        ax.tick_params(labelsize=8)
        ax.grid(alpha=.15)
        if row == 0:
            ax.set_title(rf"$\sigma={width:.3g}$", fontsize=11)
        if column == 0:
            ax.set_ylabel(rf"$A={amplitude:g}$", fontsize=11)
figure.supxlabel(r"Координата $\theta$", fontsize=13)
figure.supylabel(r"Интенсивность $|\psi|^2$; строки: амплитуда начального импульса", fontsize=13)
figure.suptitle("Профили интенсивности на сетке начальных условий", fontsize=17, y=.995)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .975), ncol=len(profile_times), frameon=False)
figure.tight_layout(rect=(.02, .02, 1, .945), h_pad=.5, w_pad=.5)
save_figure(RESULTS_DIR, "intensity_profiles_grid", figure)
plt.show()